# F1-scientific-python — Session 01: Arrays and Indexing

**Session length:** about 75 minutes • **Concepts:** numpy-arrays,
array-indexing-slicing (first half of the unit's toolkit)

This session introduces NumPy's core object — the array — and the many ways of
getting data into and out of one. Everything assumes only Calculus AB and basic
Python. Each section ends with a **Checkpoint** (do it before moving on);
all answers are collected at the end of this notebook.

In [ ]:
import numpy as np

## 1. Why arrays beat lists

Suppose you have a million numbers and you want to square every one of them.
With a plain Python list you have to write a loop (or a list comprehension), and
Python executes it one element at a time, doing type checks and bookkeeping on
every single step.

NumPy's core object is the **array**: a container of numbers that all share one
type, stored tightly packed in memory. Because of that, NumPy can run the *whole*
operation in fast, pre-compiled code. You write one line — no loop — and it is
usually tens of times faster.

There are two wins, and both matter:

1. **Speed** — the loop happens inside optimized machine code, not in Python.
2. **Clarity** — `prices * quantities` says exactly what it means. The math reads
   like math.

Let's measure the speed difference ourselves.

In [ ]:
import time

n = 1_000_000
nums_list = list(range(n))
nums_array = np.arange(n)          # an array holding 0, 1, 2, ..., 999999

t0 = time.perf_counter()
squares_list = [x * x for x in nums_list]     # plain Python: loop over every element
t1 = time.perf_counter()
squares_array = nums_array * nums_array       # NumPy: one whole-array operation
t2 = time.perf_counter()

print(f"list comprehension: {t1 - t0:.4f} seconds")
print(f"whole-array math:   {t2 - t1:.4f} seconds")
print("same first five results?", squares_list[:5] == list(squares_array[:5]))

The exact times vary by machine, but the array version is dramatically faster.

Whole-array math also makes everyday calculations shorter and easier to read:

In [ ]:
prices = np.array([2.50, 1.20, 3.00, 0.75])      # price per item
quantities = np.array([4, 10, 2, 8])             # how many of each we buy

cost_per_item = prices * quantities              # multiplies element by element
print("cost per item:", cost_per_item)
print("total cost:   ", cost_per_item.sum())

With lists, that would have been a loop and a running total. With arrays it is
two short lines that look like the arithmetic they perform.

### Checkpoint 1

1. Turn the list `[3, 1, 4, 1, 5, 9]` into an array and add 10 to every element
   in a single line, with no loop.
2. Without running it, predict the output of
   `np.array([1, 2, 3]) * np.array([4, 5, 6])`. Then run it to check.

## 2. Creating arrays

You will constantly need arrays filled with specific values: a list you already
have, all zeros, a run of consecutive integers, or evenly spaced points for a
plot. NumPy has a builder for each job.

| Builder | What it makes |
| --- | --- |
| `np.array(some_list)` | an array with the same contents as the list |
| `np.zeros(shape)` | all zeros |
| `np.ones(shape)` | all ones |
| `np.full(shape, v)` | every element equal to `v` |
| `np.arange(start, stop, step)` | like Python's `range` — stop is **excluded** |
| `np.linspace(start, stop, count)` | `count` evenly spaced values — stop is **included** |

A nested list of lists becomes a 2-D array: a grid with rows and columns.

In [ ]:
from_list = np.array([4, 8, 15, 16, 23, 42])
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])          # 2 rows, 3 columns

print("from a list:", from_list)
print("a 2-D grid:")
print(grid)

print("zeros: ", np.zeros(4))
print("ones:\n", np.ones((2, 3)))          # shape given as a tuple: (rows, columns)
print("full:  ", np.full(3, 7.5))

In [ ]:
print("arange(6):        ", np.arange(6))           # 0..5, stop excluded
print("arange(2, 12, 3): ", np.arange(2, 12, 3))    # start 2, step 3, stop excluded
print("linspace(0, 1, 5):", np.linspace(0, 1, 5))   # 5 points, both ends included

**`arange` vs `linspace`:** use `arange` when you know the *step size* and are
working with integers; use `linspace` when you know *how many* points you want —
especially for decimal values, where `linspace` avoids the rounding surprises
that decimal steps can cause in `arange`.

### Checkpoint 2

1. Create an array of every even number from 0 through 20 using `np.arange`.
2. Create exactly 9 evenly spaced values from 0 to 2 using `np.linspace`.
3. Create a 3×4 array of ones.

## 3. dtype: one shared type

Every array has a **`dtype`** (data type): the single type shared by *all* its
elements — commonly `int64` (integers), `float64` (decimals), or `bool`
(True/False). This is a big difference from lists, which mix types freely. One
shared type is exactly what makes whole-array math fast.

Two consequences you must know:

- **Mixing promotes.** Build an array from a mix of integers and decimals and
  NumPy silently upgrades everything to the more general type (`float64`).
- **Converting is explicit.** `astype` returns a new array with a new dtype —
  and converting floats to integers *chops off* the decimal part.

In [ ]:
print(np.array([1, 2, 3]).dtype)          # int64
print(np.array([1.0, 2.0]).dtype)         # float64
print(np.array([True, False]).dtype)      # bool
print(np.array([1, 2.5, 3]).dtype)        # one decimal -> everything becomes float64

floats = np.array([1.9, 2.1, 3.7])
print(floats.astype(np.int64))            # [1 2 3] — decimal parts dropped, not rounded

counts = np.array([3, 1, 4])
print(counts.astype(np.float64))          # [3. 1. 4.]

### Checkpoint 3

1. Predict the dtype of `np.array([2, 4.0, 6])`, then check. Why that type?
2. Predict the exact output of `np.array([2.7, -1.2, 9.99]).astype(np.int64)`,
   then check.

## 4. shape, ndim, size, reshape

An array's **`shape`** is a tuple giving its size along each **axis**
(direction). A 1-D array of 6 elements has shape `(6,)`. A grid with 2 rows and
3 columns has shape `(2, 3)` — rows first, then columns. Related attributes:
`ndim` (how many axes) and `size` (total element count).

**Reshaping.** `reshape` rearranges the same elements into a new shape, as long
as the total count matches. Passing `-1` for one axis asks NumPy to work that
size out for you.

In [ ]:
a = np.array([4, 8, 15, 16, 23, 42])
g = np.array([[1, 2, 3],
              [4, 5, 6]])

print("a: shape", a.shape, "| ndim", a.ndim, "| size", a.size)
print("g: shape", g.shape, "| ndim", g.ndim, "| size", g.size)

nums = np.arange(12)
print(nums.reshape(3, 4))       # 3 rows x 4 columns
print(nums.reshape(2, -1))      # 2 rows; NumPy computes 6 columns
print(nums.reshape(3, 4).reshape(-1))   # back to flat, shape (12,)

Get in the habit **now** of predicting an expression's shape before you run
it. Most NumPy bugs are shape bugs, and shape prediction is exactly what exam
multiple-choice questions test.

### Checkpoint 4

1. Guess the shape, ndim, and size of `np.array([[1, 2], [3, 4], [5, 6]])`,
   then check.
2. Reshape `np.arange(12)` into 3 rows and 4 columns, then flatten it back to
   shape `(12,)`.
3. Why does `np.arange(10).reshape(3, 4)` fail? Predict the error, then try it
   inside `try`/`except`.

## 5. Indexing and slicing basics

Getting *parts* of an array — one element, one row, a rectangular block — is
something you will do constantly.

**1-D arrays work like lists.** `a[0]`, `a[-1]`, and slices like `a[2:5]` all
behave as you expect, including step slices like `a[::2]`.

**2-D arrays take `[row, column]`.** One pair of brackets, indices separated by
a comma. A `:` in either slot means "everything along that axis".

In [ ]:
a = np.arange(10, 20)          # [10 11 12 13 14 15 16 17 18 19]
print("a[0] =", a[0], "  a[-1] =", a[-1], "  a[2:5] =", a[2:5], "  a[::2] =", a[::2])

g = np.arange(20).reshape(4, 5)
print("g:")
print(g)
print("g[1, 3]   =", g[1, 3])       # row 1, column 3
print("g[1]      =", g[1])          # entire row 1
print("g[:, 2]   =", g[:, 2])       # entire column 2
print("g[0:2, 1:4] =")              # rows 0-1, columns 1-3: a block
print(g[0:2, 1:4])

### Checkpoint 5

Using `g = np.arange(20).reshape(4, 5)`:

1. Extract the last row, the last column, and the 2×2 block in the bottom-right
   corner.
2. Predict the shape of `g[1:3, ::2]`, then check.

## 6. Boolean masks

Comparing an array to a value produces an array of True/False — one answer per
element. Using that **mask** inside square brackets keeps only the elements
where the mask is True. Combine masks with `&` (and), `|` (or), `~` (not) —
each comparison wrapped in parentheses. You can also *assign* through a mask.

In [ ]:
temps = np.array([12, 31, 24, 8, 27, 35, 19])

mask = temps > 20
print("mask:         ", mask)
print("temps > 20:   ", temps[mask])
print("20 < t < 32:  ", temps[(temps > 20) & (temps < 32)])

capped = temps.copy()
capped[capped > 30] = 30            # assign through a mask
print("capped at 30: ", capped)

Two counting idioms fall straight out of masks, because True counts as 1 and
False as 0:

- `mask.sum()` — **how many** elements pass the test;
- `mask.mean()` — **what fraction** of elements pass the test.

In [ ]:
print("how many above 20:", (temps > 20).sum())
print("fraction above 20:", (temps > 20).mean())

### Checkpoint 6

Using the same `temps` array:

1. Select the readings that are below 10 **or** above 30.
2. How many readings are between 15 and 30 (inclusive)? Answer with one
   expression, no loop.

## 7. Integer-array indexing

A list (or array) of indices inside the brackets picks out exactly those
positions, in exactly that order — repeats allowed. On a 2-D array, one list of
row indices selects whole rows; a *pair* of lists selects individual
`(row, column)` elements pairwise.

In [ ]:
a = np.arange(10, 20)
print("picked:", a[[7, 0, 3, 3]])        # positions 7, 0, 3, 3 in that order

g = np.arange(20).reshape(4, 5)
print("rows 0 and 2:")
print(g[[0, 2]])
print("elements (0,1) and (3,4):", g[[0, 3], [1, 4]])   # pairwise: g[0,1], g[3,4]

### Checkpoint 7

Using `g = np.arange(20).reshape(4, 5)`:

1. Use integer-array indexing to pull out rows 2 and 0, in that order.
2. In a single expression, fetch the three elements at (0, 0), (1, 2), and
   (3, 4).

## 8. Common pitfalls I

Two traps that catch every newcomer. Meeting them here, on purpose, is cheaper
than meeting them in a graded task.

**Pitfall — a slice is a view, not a copy.** Slicing does not copy data; it
gives you a window onto the *same* memory. Modify the slice and you modify the
original:

In [ ]:
g = np.arange(12).reshape(3, 4)
first_row = g[0]              # BROKEN as a "backup": this is a view
first_row[:] = 0
print(g)                      # surprise: row 0 of g is gone too

g = np.arange(12).reshape(3, 4)
first_row = g[0].copy()       # FIX: an independent copy
first_row[:] = 0
print(g)                      # g is untouched

(Boolean-mask and integer-array indexing *do* return copies; it is plain
slices that share memory.)

**Pitfall — integer arrays truncate.** An array's dtype is fixed at creation.
Assign a decimal into an integer array and the decimal part is silently chopped
off:

In [ ]:
a = np.arange(5)              # dtype int64
a[2] = 3.9                    # BROKEN: silently stored as 3
print(a)

b = np.arange(5).astype(np.float64)   # FIX: make it a float array first
b[2] = 3.9
print(b)

### Checkpoint 8

1. After `g = np.arange(6).reshape(2, 3)`, `top = g[0]`, `top[0] = 99` — what
   does `g` now contain? Rewrite the second line so `g` is protected.
2. `z = np.zeros(3)` then `z[0] = 0.5` — does this store 0.5 or 0? Why is this
   case different from the `np.arange(5)` example above?

## 9. Worked exam-style example: predict the output

Round 1 loves five-option multiple choice where you trace indexing by hand.
Here is one, solved step by step in the exam's register:

> Let `g = np.arange(20).reshape(4, 5)`. What is the value of
> `g[g % 3 == 0][2]`?
>
> (A) 2  (B) 3  (C) 6  (D) 9  (E) an error is raised

1. `g` holds 0–19 in 4 rows of 5.
2. `g % 3 == 0` is a mask over the whole grid, True at multiples of 3:
   0, 3, 6, 9, 12, 15, 18.
3. Mask indexing on a 2-D array returns a **1-D** array of the kept values, in
   row-by-row reading order: `[0, 3, 6, 9, 12, 15, 18]`.
4. Index `[2]` picks the third: **6 → answer (C)**.

Verify (allowed here, not in the exam room):

In [ ]:
g = np.arange(20).reshape(4, 5)
print(g[g % 3 == 0][2])

### Checkpoint 9

1. Exam-style, by hand first: `np.linspace(0, 1, 5)[[0, -1]]` — what array
   does it produce? (A) `[0. 1.]` (B) `[0. 0.75]` (C) `[0.25 1.]`
   (D) `[0. 0.25]` (E) an error. Write out your two reasoning steps.
2. By hand: `np.arange(10, 20)[::3].size` — what integer, and why?

## Checkpoint answers

Worked answers to every checkpoint in this session. Try each honestly before
peeking.

### Checkpoint 1 answers

In [ ]:
print(np.array([3, 1, 4, 1, 5, 9]) + 10)                 # 1.
print(np.array([1, 2, 3]) * np.array([4, 5, 6]))         # 2. elementwise: [4 10 18]

### Checkpoint 2 answers

In [ ]:
print(np.arange(0, 21, 2))        # 1. stop must be past 20 so 20 is included
print(np.linspace(0, 2, 9))       # 2. exactly 9 points, both endpoints included
print(np.ones((3, 4)))            # 3. shape as a tuple

### Checkpoint 3 answers

In [ ]:
# 1. float64 — the single 4.0 forces the shared type up to decimals
print(np.array([2, 4.0, 6]).dtype)
# 2. [2 -1 9] — astype to int64 chops the decimal part (no rounding)
print(np.array([2.7, -1.2, 9.99]).astype(np.int64))

### Checkpoint 4 answers

In [ ]:
arr = np.array([[1, 2], [3, 4], [5, 6]])
print(arr.shape, arr.ndim, arr.size)      # 1. (3, 2), 2, 6

r = np.arange(12).reshape(3, 4)           # 2.
print(r)
print(r.reshape(-1))

try:                                       # 3. 10 elements cannot fill 3x4 = 12 slots
    np.arange(10).reshape(3, 4)
except ValueError as err:
    print("ValueError:", err)

### Checkpoint 5 answers

In [ ]:
g = np.arange(20).reshape(4, 5)
print(g[-1])              # 1. last row
print(g[:, -1])           #    last column
print(g[2:, 3:])          #    bottom-right 2x2 block
print(g[1:3, ::2].shape)  # 2. rows 1-2 and columns 0,2,4 -> (2, 3)

### Checkpoint 6 answers

In [ ]:
temps = np.array([12, 31, 24, 8, 27, 35, 19])
print(temps[(temps < 10) | (temps > 30)])            # 1.
print(((temps >= 15) & (temps <= 30)).sum())         # 2.

### Checkpoint 7 answers

In [ ]:
g = np.arange(20).reshape(4, 5)
print(g[[2, 0]])                      # 1. rows in the order given
print(g[[0, 1, 3], [0, 2, 4]])        # 2. pairwise picks

### Checkpoint 8 answers

In [ ]:
# 1. g becomes [[99, 1, 2], [3, 4, 5]] — top is a view of row 0.
#    Fix: top = g[0].copy()
g = np.arange(6).reshape(2, 3)
top = g[0].copy()
top[0] = 99
print(g)

# 2. It stores 0.5. np.zeros makes a float64 array, so decimals fit; the
#    truncation trap only bites integer-dtype arrays like np.arange(5).
z = np.zeros(3)
z[0] = 0.5
print(z, z.dtype)

### Checkpoint 9 answers

In [ ]:
# 1. Step 1: linspace(0, 1, 5) = [0, 0.25, 0.5, 0.75, 1].
#    Step 2: integer-array indexing picks positions 0 and -1 -> [0. 1.] -> (A)
print(np.linspace(0, 1, 5)[[0, -1]])

# 2. [::3] keeps positions 0, 3, 6, 9 of a 10-element array -> size 4
print(np.arange(10, 20)[::3].size)